In [1]:
from collections import deque  # deque is useful for the BFS queue


class SearchAlgorithms:
    """A small class containing the search methods for this lab."""

    def __init__(self):
        # The successor order is exactly the order given in the lab manual.
        self.graph = {
            "A": ["B", "C"],
            "B": ["D", "E"],
            "C": ["F"],
            "D": [],
            "E": ["G"],
            "F": [],
            "G": []
        }

        # This is the weighted extension required for the UCS task.
        # C -> G is the added corridor from Section 3.7.
        self.weighted_graph = {
            "A": [("B", 1), ("C", 6)],
            "B": [("D", 1), ("E", 1)],
            "C": [("F", 1), ("G", 6)],
            "D": [],
            "E": [("G", 1)],
            "F": [],
            "G": []
        }

    def bfs(self, start, goal):
        """Breadth-first search: it explores one level at a time."""
        queue = deque([(start, [start])])  # store node and path together
        visited = {start}                 # avoid adding a node twice
        order = []                        # stores processing order

        while queue:
            node, path = queue.popleft()  # remove from the front
            order.append(node)            # count this node as processed

            if node == goal:              # stop when the goal is processed
                return path, order

            for neighbour in self.graph[node]:
                if neighbour not in visited:
                    visited.add(neighbour)
                    new_path = path + [neighbour]
                    queue.append((neighbour, new_path))

        return None, order                 # goal was not found

    def dfs(self, start, goal):
        """Depth-first search: it follows one branch before backtracking."""
        stack = [(start, [start])]        # the last item is the stack top
        visited = set()                   # processed nodes
        order = []

        while stack:
            node, path = stack.pop()      # remove from the top

            if node in visited:
                continue

            visited.add(node)
            order.append(node)

            if node == goal:
                return path, order

            # Reverse insertion keeps the first listed neighbour on top.
            for neighbour in reversed(self.graph[node]):
                if neighbour not in visited:
                    new_path = path + [neighbour]
                    stack.append((neighbour, new_path))

        return None, order

    def dls(self, start, goal, limit):
        """Depth-limited DFS returning FOUND, CUTOFF, or FAILURE."""
        order = []

        def visit(node, path, depth):
            order.append(node)

            if node == goal:
                return "FOUND", path

            if depth == limit:
                # There may be more nodes below this point.
                return "CUTOFF", None

            cutoff_found = False

            for neighbour in self.graph[node]:
                # This is a path-local cycle check.
                if neighbour in path:
                    continue

                result, result_path = visit(
                    neighbour, path + [neighbour], depth + 1
                )

                if result == "FOUND":
                    return result, result_path

                if result == "CUTOFF":
                    cutoff_found = True

            if cutoff_found:
                return "CUTOFF", None
            return "FAILURE", None

        status, path = visit(start, [start], 0)  # root has depth 0
        return status, path, order

    def iddfs(self, start, goal, maximum_depth):
        """Run DLS repeatedly with increasing depth limits."""
        attempts = []

        for limit in range(maximum_depth + 1):
            result = self.dls(start, goal, limit)
            status, path, order = result
            attempts.append((limit, status, path, order))

            if status == "FOUND":
                return status, path, attempts

        return attempts[-1][1], None, attempts

    def ucs(self, start, goal):
        """Uniform-cost search using a simple list as a priority queue."""
        # Each item is [total_cost, node, path].
        frontier = [[0, start, [start]]]
        best_cost = {start: 0}
        order = []

        while frontier:
            # Find the item having the smallest total cost.
            cheapest_item = frontier[0]
            for item in frontier:
                if item[0] < cheapest_item[0]:
                    cheapest_item = item

            frontier.remove(cheapest_item)
            cost, node, path = cheapest_item

            # Ignore an older, more expensive copy of the same node.
            if cost != best_cost[node]:
                continue

            order.append((node, cost))

            if node == goal:
                return path, cost, order

            for neighbour, edge_cost in self.weighted_graph[node]:
                new_cost = cost + edge_cost

                if (neighbour not in best_cost or
                        new_cost < best_cost[neighbour]):
                    best_cost[neighbour] = new_cost
                    new_path = path + [neighbour]
                    frontier.append([new_cost, neighbour, new_path])

        return None, None, order

    def calculate_cost(self, path):
        """Add the edge costs of a path independently."""
        total = 0

        for first, second in zip(path, path[1:]):
            for neighbour, cost in self.weighted_graph[first]:
                if neighbour == second:
                    total += cost
                    break

        return total

    def bfs_on_weighted_graph(self, start, goal):
        """BFS on the weighted graph while ignoring the weights."""
        simple_graph = {}

        for node in self.weighted_graph:
            # Keep only neighbour names for BFS.
            simple_graph[node] = []
            for neighbour, cost in self.weighted_graph[node]:
                simple_graph[node].append(neighbour)

        queue = deque([(start, [start])])
        visited = {start}
        order = []

        while queue:
            node, path = queue.popleft()
            order.append(node)

            if node == goal:
                return path, order

            for neighbour in simple_graph[node]:
                if neighbour not in visited:
                    visited.add(neighbour)
                    queue.append((neighbour, path + [neighbour]))

        return None, order


# ------------------------- Main program -------------------------
search = SearchAlgorithms()  # create one object from our class

print("TASK 1: BFS")
bfs_path, bfs_order = search.bfs("A", "G")
print("Visit order:", bfs_order)
print("Path:", bfs_path)

print("\nTASK 1: DFS")
dfs_path, dfs_order = search.dfs("A", "G")
print("Visit order:", dfs_order)
print("Path:", dfs_path)

print("\nTASK 2: DLS")
for limit in [1, 2, 3, 4]:
    status, path, order = search.dls("A", "G", limit)
    print("Limit:", limit, "Status:", status,
          "Path:", path, "Order:", order)

print("\nTASK 2: IDDFS")
for maximum_depth in [1, 2, 3, 4]:
    status, path, attempts = search.iddfs("A", "G", maximum_depth)
    summary = []
    for limit, result, found_path, order in attempts:
        summary.append((limit, result))
    print("Maximum depth:", maximum_depth,
          "Status:", status, "Path:", path,
          "Attempts:", summary)

print("\nTASK 3: BFS on weighted graph")
bfs_weighted_path, bfs_weighted_order = search.bfs_on_weighted_graph("A", "G")
print("Path:", bfs_weighted_path)
print("Hops:", len(bfs_weighted_path) - 1)
print("Cost:", search.calculate_cost(bfs_weighted_path))
print("Processing order:", bfs_weighted_order)

print("\nTASK 3: UCS")
ucs_path, ucs_cost, ucs_order = search.ucs("A", "G")
print("Path:", ucs_path)
print("Hops:", len(ucs_path) - 1)
print("UCS cost:", ucs_cost)
print("Independent cost check:", search.calculate_cost(ucs_path))
print("Processing order:", ucs_order)

print("\nEDGE CASE: start equals goal")
print(search.bfs("A", "A"))

TASK 1: BFS
Visit order: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Path: ['A', 'B', 'E', 'G']

TASK 1: DFS
Visit order: ['A', 'B', 'D', 'E', 'G']
Path: ['A', 'B', 'E', 'G']

TASK 2: DLS
Limit: 1 Status: CUTOFF Path: None Order: ['A', 'B', 'C']
Limit: 2 Status: CUTOFF Path: None Order: ['A', 'B', 'D', 'E', 'C', 'F']
Limit: 3 Status: FOUND Path: ['A', 'B', 'E', 'G'] Order: ['A', 'B', 'D', 'E', 'G']
Limit: 4 Status: FOUND Path: ['A', 'B', 'E', 'G'] Order: ['A', 'B', 'D', 'E', 'G']

TASK 2: IDDFS
Maximum depth: 1 Status: CUTOFF Path: None Attempts: [(0, 'CUTOFF'), (1, 'CUTOFF')]
Maximum depth: 2 Status: CUTOFF Path: None Attempts: [(0, 'CUTOFF'), (1, 'CUTOFF'), (2, 'CUTOFF')]
Maximum depth: 3 Status: FOUND Path: ['A', 'B', 'E', 'G'] Attempts: [(0, 'CUTOFF'), (1, 'CUTOFF'), (2, 'CUTOFF'), (3, 'FOUND')]
Maximum depth: 4 Status: FOUND Path: ['A', 'B', 'E', 'G'] Attempts: [(0, 'CUTOFF'), (1, 'CUTOFF'), (2, 'CUTOFF'), (3, 'FOUND')]

TASK 3: BFS on weighted graph
Path: ['A', 'C', 'G']
Hops: 2
Cost: 12